# M39 lab solution: two servers, one client

**Brief.** Two servers, one client: put your retriever and a web-search tool in separate MCP servers and route between them from one client.

- [`retriever_server.py`](retriever_server.py): `larkspur-handbook`, one tool `search_handbook`, the offline TF-IDF retriever over the handbook (the M38 lab's server, rebuilt here so the lab stands alone).
- [`web_server.py`](web_server.py): `web-search`, one tool `search_web`, Tavily search.
- This notebook is one client holding a session to each server, as the multi-server client of lesson 39.4 does. It merges both tool lists, remembers which session owns each tool, and lets the model choose. The routing is the model's tool choice; the client only forwards the call to the right session.

Five questions: three the handbook answers and two it cannot (current public facts). The routing printed for each is the list of tools the model called.

## Setup

In [ ]:
import asyncio
import os
import sys
import time
from contextlib import AsyncExitStack
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
nest_asyncio.apply()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_experientiallabs_llm  # noqa: E402
from lab_kit import load_jsonl  # noqa: E402

golden = {g["id"]: g for g in load_jsonl(Path("../data/larkspur/golden.jsonl"))}
llm = get_experientiallabs_llm()
MODEL = llm.model_name

HERE = str(Path(".").resolve())
SERVERS = {
    "larkspur-handbook": StdioServerParameters(command=sys.executable, args=["retriever_server.py"], cwd=HERE),
    # A stdio server starts with a minimal environment, so pass the one key the web server needs.
    "web-search": StdioServerParameters(
        command=sys.executable, args=["web_server.py"], cwd=HERE,
        env={"PATH": os.environ.get("PATH", ""), "TAVILY_API_KEY": os.environ.get("TAVILY_API_KEY", "")},
    ),
}

QUESTIONS = [
    (golden["g04"]["question"], "search_handbook"),
    (golden["g10"]["question"], "search_handbook"),
    (golden["g22"]["question"], "search_handbook"),
    ("What is the latest stable release of the Python programming language?", "search_web"),
    ("Which organisation maintains the Model Context Protocol specification?", "search_web"),
]
print("model:", MODEL)

## The client: one session per server, one merged tool list

In [ ]:
SYSTEM = ("You are Larkspur Cloud's support assistant with two tools. search_handbook searches Larkspur's own "
          "customer handbook; search_web searches the public web. Use the handbook for anything about Larkspur and "
          "the web for anything else. Answer only from tool results and name the source in brackets.")
MAX_TOOL_ROUNDS = 4


async def run_questions(questions: list[tuple[str, str]]) -> list[dict]:
    out = []
    async with AsyncExitStack() as stack:
        errlog = stack.enter_context(open(os.devnull, "w"))
        owner, specs = {}, []
        for name, params in SERVERS.items():
            read, write = await stack.enter_async_context(stdio_client(params, errlog=errlog))
            session = await stack.enter_async_context(ClientSession(read, write))
            await session.initialize()
            for t in (await session.list_tools()).tools:
                owner[t.name] = (name, session)
                specs.append({"type": "function", "function": {"name": t.name, "description": t.description,
                                                               "parameters": t.inputSchema}})
        print("tools:", {tool: server for tool, (server, _) in owner.items()})
        bound = llm.bind_tools(specs)

        for question, expected in questions:
            messages = [SystemMessage(SYSTEM), HumanMessage(question)]
            routed = []
            start = time.perf_counter()
            for _ in range(MAX_TOOL_ROUNDS):
                reply = await bound.ainvoke(messages)
                messages.append(reply)
                if not reply.tool_calls:
                    break
                for call in reply.tool_calls:
                    server, session = owner[call["name"]]
                    routed.append(f"{call['name']}@{server}")
                    result = await session.call_tool(call["name"], arguments=call["args"])
                    text = "\n".join(c.text for c in result.content if getattr(c, "text", None))
                    messages.append(ToolMessage(content=text, tool_call_id=call["id"]))
            out.append({"question": question, "expected": expected, "routed": routed,
                        "answer": messages[-1].content, "seconds": round(time.perf_counter() - start, 2)})
    return out


results = asyncio.run(run_questions(QUESTIONS))
for r in results:
    print(f"Q: {r['question']}\n  routed: {r['routed'] or 'no tool'} (expected {r['expected']})\n  answer: {r['answer'][:300]}\n")

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df["first_tool"] = df["routed"].map(lambda r: r[0].split("@")[0] if r else "none")
df["routed_as_expected"] = df["first_tool"] == df["expected"]
print("model:", MODEL, "| run:", time.strftime("%Y-%m-%d"))
print(df[["question", "expected", "routed", "routed_as_expected", "seconds"]].to_markdown(index=False))
print(f"\nrouted as expected: {df['routed_as_expected'].sum()} of {len(df)}")

## What this shows

- Neither server knows the other exists. The client is the only place that sees both tool lists, so routing, naming clashes and per-server permissions are all client decisions.
- The model routes on tool descriptions. If a Larkspur question goes to `search_web`, sharpen the two descriptions before touching the prompt.
- A stdio server inherits only the environment the client gives it: the web server gets `TAVILY_API_KEY` and `PATH`, the retriever gets only the SDK's default variables (such as `HOME` and `PATH`), and neither sees the model key.